# Particles stuck in the grid - horizontal wall or seafloor?

The runs use `AdvectionRK4_3D` with no anti-beaching kernel (the only deletion
kernel is `CheckError`, on error state). Particles that reach a masked cell
stop and stay: ~48% of completed trajectories show no movement over their last
30 days, concentrated in the E groups.

This asks *how* they are stuck, the way the supervisor framed it:

* **Vertical (seafloor):** the particle sinks onto the bottom - its depth
  reaches the local seafloor and it cannot go deeper.
* **Horizontal (land wall):** the particle is pinned laterally against a land
  cell while still well above the bottom.

The test uses the model's OWN land-sea mask and bathymetry - the same
`FieldSet` grid `AdvectionRK4_3D` integrates in - rather than an external
coastline. `mbathy` (number of wet levels per column) with `gdepw_0` gives the
seafloor depth of every cell; a cell with `mbathy == 0` is land.

(Direct re-sampling of the FieldSet velocity outside a running ParticleSet is
unreliable in Parcels 3.1.2 for C-grids - it was tested and returns zero even
in the open ocean - so the geometric mask, which is what the kernel actually
sees, is the robust discriminator.)


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import xarray as xr
from scipy.spatial import cKDTree
from joblib import Parallel, delayed
import config as C

# --- TUNE ME -----------------------------------------------------------------
N_SHOW      = 5          # example particles per group for the 3D plot
GROUPS_E    = ["E1", "E0", "E2"]
BOTTOM_TOL_M = 5.0       # within this of the seafloor counts as "grounded"
N_POP        = 8000      # frozen particles sampled for the population split
# -----------------------------------------------------------------------------

MESH_H = "/work/bk1450/b383184/Amazon/Mercator/data/Hgr_cmesh.nc"
MESH_Z = "/work/bk1450/b383184/Amazon/Mercator/data/Zgr_cmesh2.nc"
N_IO   = int(os.environ.get("SLURM_CPUS_PER_TASK", 16))
inv = {v: k for k, v in C.GROUP_NAMES.items()}
store_paths = {p.stem: p for p in C.list_stores()}
print("groups:", GROUPS_E, "| ids:", [inv[g] for g in GROUPS_E])

## Load the model grid - the FieldSet's land mask and bathymetry
`mbathy` and the mesh come straight from the files the Parcels run passed to
`FieldSet.from_netcdf`. `bottom_depth[j,i]` is the seafloor; `is_land[j,i]`
marks dry columns. Particles are on T-cells, so we locate them on the T-point
coordinates `glamt`/`gphit`.

In [ ]:
Zg = xr.open_dataset(MESH_Z)
Hg = xr.open_dataset(MESH_H)
mbathy = np.asarray(Zg["mbathy"]).squeeze().astype(int)      # wet levels per column
gdepw  = np.asarray(Zg["gdepw_0"]).squeeze()                 # w-level depths (m)
glamt  = np.asarray(Hg["glamt"]).squeeze()                   # T-point lon
gphit  = np.asarray(Hg["gphit"]).squeeze()                   # T-point lat

bottom_depth = np.where(mbathy > 0, gdepw[np.clip(mbathy, 0, len(gdepw) - 1)],
                        np.nan)                              # seafloor depth (m)
is_land = mbathy == 0
JI = glamt.shape
grid_tree = cKDTree(np.c_[glamt.ravel(), gphit.ravel()])

def locate(lat, lon):
    _, idx = grid_tree.query(np.c_[lon, lat])
    return idx                                              # flat T-cell index

def bottom_at(lat, lon):
    return np.atleast_1d(bottom_depth.ravel()[locate(lat, lon)])

def land_neighbour(lat, lon):
    # is any of the 8 neighbours of the particle's T-cell dry?
    idx = locate(lat, lon); out = np.zeros(len(np.atleast_1d(idx)), bool)
    for n, ix in enumerate(np.atleast_1d(idx)):
        j, i = np.unravel_index(ix, JI)
        js = slice(max(j - 1, 0), j + 2); iss = slice(max(i - 1, 0), i + 2)
        out[n] = is_land[js, iss].any()
    return out

print(f"grid {JI} | wet columns {int((~is_land).sum()):,} | land {int(is_land.sum()):,}")
print(f"seafloor depth range (wet): {np.nanmin(bottom_depth):.1f} .. "
      f"{np.nanmax(bottom_depth):.0f} m")

## Pick 5 particles per group and stream their full paths

In [ ]:
lab = pd.read_parquet(C.LABELED_FILE,
                      columns=["trajectory_id", "store", "cluster_group", "status"])
lab = lab[(lab.status == "complete") & (lab.cluster_group >= 0)]

pick = pd.concat([lab[lab.cluster_group == inv[g]]
                  .sample(N_SHOW, random_state=C.RANDOM_STATE).assign(grp=g)
                  for g in GROUPS_E], ignore_index=True)
pick["local"] = pick.trajectory_id % C.TRAJ_PER_STORE

def _full(name, grp):
    ds = xr.open_zarr(store_paths[name]); out = {}
    for l, tid in zip(grp.local.to_numpy(), grp.trajectory_id.to_numpy()):
        lo, la, z = ds.lon.values[l], ds.lat.values[l], ds.z.values[l]
        t = ds.time.values[l]
        m = ~np.isnan(lo) & ~np.isnat(t)
        out[int(tid)] = dict(lon=lo[m], lat=la[m], z=z[m],
                             day=(t[m] - t[m][0]) / np.timedelta64(1, "D"))
    return out

paths = {}
for d in Parallel(n_jobs=N_IO, backend="threading")(
        delayed(_full)(n, g) for n, g in pick.groupby("store")):
    paths.update(d)
print(f"streamed {len(paths)} example trajectories")

## 3D view - drag to rotate
Longitude x latitude x depth (down). Each track is coloured by group; the black
marker is the final position, and a red stem drops from it to the local
seafloor. If the stem has zero length the particle is **on the bottom**
(vertical); if the track's lon/lat freeze while it sits well above the seafloor
it is against a **land wall** (horizontal).

In [21]:
import plotly.graph_objects as go
COL = {"E1": "#2a78d6", "E0": "#eb6834", "E2": "#1baf7a"}

# --- cheap bathymetry surface over the particles' bounding box (transparent) ---
_allon = np.concatenate([paths[int(t)]["lon"] for t in pick.trajectory_id])
_allat = np.concatenate([paths[int(t)]["lat"] for t in pick.trajectory_id])
gx = np.linspace(_allon.min() - .5, _allon.max() + .5, 60)   # coarse -> cheap
gy = np.linspace(_allat.min() - .5, _allat.max() + .5, 60)
GX, GY = np.meshgrid(gx, gy)
_, gidx = grid_tree.query(np.c_[GX.ravel(), GY.ravel()])
GZ = bottom_depth.ravel()[gidx].reshape(GX.shape)            # seafloor; NaN over land

# cap the depth axis just below the deepest particle so the shelf grounding is
# visible; hide surface cells deeper than the cap (open ocean off the shelf)
_zmax = max(np.nanmax([paths[int(t)]["z"].max() for t in pick.trajectory_id]), 20)
ZCAP = min(100.0, _zmax + 15)
GZ = np.where(GZ <= ZCAP, GZ, np.nan)

fig = go.Figure()
fig.add_trace(go.Surface(x=GX, y=GY, z=GZ, opacity=0.30, showscale=False,
                         colorscale="dense", hoverinfo="skip", name="seafloor"))
for _, r in pick.iterrows():
    p = paths[int(r.trajectory_id)]
    fig.add_trace(go.Scatter3d(
        x=p["lon"], y=p["lat"], z=p["z"], mode="lines",
        line=dict(color=COL[r.grp], width=4), name=r.grp,
        legendgroup=r.grp, showlegend=False,
        customdata=np.c_[p["day"]],
        hovertemplate="lon %{x:.2f}  lat %{y:.2f}<br>depth %{z:.1f} m"
                      "<br>day %{customdata[0]:.0f}<extra></extra>"))
    xf, yf, zf = p["lon"][-1], p["lat"][-1], p["z"][-1]
    Hb = float(bottom_at(yf, xf)[0])
    fig.add_trace(go.Scatter3d(x=[xf, xf], y=[yf, yf], z=[zf, Hb], mode="lines",
                               line=dict(color="red", width=3), showlegend=False,
                               hovertemplate=f"seafloor {Hb:.1f} m<extra></extra>"))
    fig.add_trace(go.Scatter3d(x=[xf], y=[yf], z=[zf], mode="markers",
                               marker=dict(size=4, color="black"), showlegend=False))
for g in GROUPS_E:
    fig.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode="lines",
                               line=dict(color=COL[g], width=4), name=g))
fig.update_layout(
    title="E-group particles: track (colour) with stem to the model seafloor (red)",
    scene=dict(xaxis_title="longitude", yaxis_title="latitude",
               zaxis_title="depth (m)", zaxis=dict(range=[ZCAP, 0]),      # depth down, capped at the shelf
               aspectmode="cube",
               camera=dict(eye=dict(x=1.5, y=-1.6, z=0.7))),
    height=760, margin=dict(l=0, r=0, t=40, b=0),
    legend=dict(orientation="h", y=-0.02))
fig.write_html("particles_stuck_grid_3d_2.html", include_plotlyjs=True)
print("wrote particles_stuck_grid_3d_2.html")
fig.show()

wrote particles_stuck_grid_3d_2.html


## Classify the 15 examples
For each: local seafloor depth, final depth, gap to the bottom, and whether a
neighbouring cell is land.

In [22]:
rows = []
for _, r in pick.iterrows():
    p = paths[int(r.trajectory_id)]
    yf, xf, zf = p["lat"][-1], p["lon"][-1], p["z"][-1]
    Hb = float(bottom_at(yf, xf))
    grounded = zf >= Hb - BOTTOM_TOL_M
    wall = bool(land_neighbour(yf, xf)[0])
    kind = "VERTICAL (seafloor)" if grounded else (
           "HORIZONTAL (land wall)" if wall else "interior?")
    rows.append(dict(grp=r.grp, tid=int(r.trajectory_id),
                     final_depth=round(zf, 1), seafloor=round(Hb, 1),
                     gap_to_bottom=round(Hb - zf, 1), land_nbr=wall, kind=kind))
ex = pd.DataFrame(rows).sort_values(["grp", "tid"])
print(ex.to_string(index=False))

grp      tid  final_depth  seafloor  gap_to_bottom  land_nbr                   kind
 E0  1523916         43.7      31.8          -11.9     False    VERTICAL (seafloor)
 E0  2910988         43.7      43.7           -0.0     False    VERTICAL (seafloor)
 E0  8567697         37.3      37.3           -0.0     False    VERTICAL (seafloor)
 E0 13381849         43.7      43.7           -0.0     False    VERTICAL (seafloor)
 E0 14175711         27.2      27.2           -0.0     False    VERTICAL (seafloor)
 E1  3082837         12.4      12.4           -0.0      True    VERTICAL (seafloor)
 E1  8563355         43.7      43.7           -0.0     False    VERTICAL (seafloor)
 E1  9094750         10.5       8.7           -1.7     False    VERTICAL (seafloor)
 E1 12671016         20.0      20.0           -0.0      True    VERTICAL (seafloor)
 E1 14182384         10.5      10.5            0.0     False    VERTICAL (seafloor)
 E2   638273          0.0       8.7            8.7      True HORIZONTAL (lan

/tmp/ipykernel_1236836/3393755947.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Hb = float(bottom_at(yf, xf))


## Population split - of all stuck particles, how many bottom vs wall?
"Stuck" uses the same proxy as before: position unchanged (< 1 km) between the
last two sampled days, over the whole archive. That fraction is then split into
seafloor vs wall by streaming each stuck particle's final depth and comparing it
to the model seafloor.

In [23]:
d0, d1 = C.DAYS[-2], C.DAYS[-1]
_cols = ["trajectory_id", "store", "cluster_group", "status",
         f"lat_{d0}", f"lon_{d0}", f"lat_{d1}", f"lon_{d1}"]
full = pd.read_parquet(C.LABELED_FILE, columns=_cols)
full = full[(full.status == "complete") & (full.cluster_group >= 0)]
mvkm = np.hypot(full[f"lat_{d1}"] - full[f"lat_{d0}"],
                (full[f"lon_{d1}"] - full[f"lon_{d0}"])
                * np.cos(np.deg2rad(full[f"lat_{d0}"]))) * 111.32
full["frozen"] = mvkm < 1.0
n_all = len(full); n_frozen = int(full.frozen.sum())
print(f"stuck (any kind): {n_frozen:,} of {n_all:,} = {100*n_frozen/n_all:.1f}%")

stuck = full[full.frozen].sample(min(N_POP, n_frozen), random_state=C.RANDOM_STATE).copy()
stuck["local"] = stuck.trajectory_id % C.TRAJ_PER_STORE

def _finaldepth(name, grp):
    ds = xr.open_zarr(store_paths[name]); out = []
    for l, tid in zip(grp.local.to_numpy(), grp.trajectory_id.to_numpy()):
        lo, la, z = ds.lon.values[l], ds.lat.values[l], ds.z.values[l]
        m = ~np.isnan(lo)
        out.append((int(tid), la[m][-1], lo[m][-1], z[m][-1]))
    return out

fin = [x for part in Parallel(n_jobs=N_IO, backend="threading")(
           delayed(_finaldepth)(n, g) for n, g in stuck.groupby("store")) for x in part]
fin = pd.DataFrame(fin, columns=["trajectory_id", "lat", "lon", "z"]).merge(
    stuck[["trajectory_id", "cluster_group"]], on="trajectory_id")
fin["seafloor"] = bottom_at(fin.lat.to_numpy(), fin.lon.to_numpy())
fin["grounded"] = fin.z >= fin.seafloor - BOTTOM_TOL_M
fin["land_nbr"] = land_neighbour(fin.lat.to_numpy(), fin.lon.to_numpy())
fin["kind"] = np.where(fin.grounded, "vertical (seafloor)",
                       np.where(fin.land_nbr, "horizontal (wall)", "interior?"))
print(f"\nof {len(fin):,} sampled stuck particles:")
vc = fin.kind.value_counts()
for k, n in vc.items():
    print(f"  {k:22s} {n:7,}  {100*n/len(fin):5.1f}%")

fr = 100 * n_frozen / n_all
print(f"\n=> as a share of ALL {n_all:,} completed trajectories:")
for k, n in vc.items():
    print(f"   {k:22s} {fr*n/len(fin):5.1f}%")

stuck (any kind): 7,310,647 of 15,266,266 = 47.9%

of 8,000 sampled stuck particles:
  vertical (seafloor)      6,987   87.3%
  horizontal (wall)          993   12.4%
  interior?                   20    0.2%

=> as a share of ALL 15,266,266 completed trajectories:
   vertical (seafloor)     41.8%
   horizontal (wall)        5.9%
   interior?                0.1%


## Per-group breakdown

In [24]:
tab = (fin.assign(grp=fin.cluster_group.map(C.GROUP_NAMES))
          .groupby("grp").agg(n=("z", "size"),
                              seafloor_med=("seafloor", "median"),
                              depth_med=("z", "median"),
                              pct_vertical=("grounded", lambda s: 100 * s.mean()),
                              pct_land_nbr=("land_nbr", lambda s: 100 * s.mean())))
tab = tab[tab.n >= 20].sort_values("pct_vertical", ascending=False)
print(tab.round(1).to_string())
OUT = C.DATA_DIR / "stuck_classification.parquet"
fin.to_parquet(OUT, index=False)
print("\nsaved", fin.shape, "->", OUT)

        n  seafloor_med  depth_med  pct_vertical  pct_land_nbr
grp                                                           
E1   1370          12.4       17.1          99.9           7.5
S     213          23.3       23.3          99.5          17.8
E0    515          37.3       37.3          97.1          19.4
S1     54          37.3       37.3          87.0          31.5
E2   5817           8.7       10.5          83.4          25.5

saved (8000, 9) -> /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_30_180_stdZ_w/data/stuck_classification.parquet
